In [51]:
from typing import Dict, TypedDict
from langgraph.graph import StateGraph, END, START
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.tools import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from IPython.display import display, Image, Markdown
from langchain_core.runnables.graph import MermaidDrawMethod
from dotenv import load_dotenv
import os

load_dotenv()
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
os.environ["Tavily_API_KEY"] = os.getenv("Tavily_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash",
                             verbose=True,
                             temperature=0.5,
                             api_key=GOOGLE_API_KEY)




In [52]:
llm.invoke("hiu, what is you name?!")

AIMessage(content="I am a large language model, trained by Google. I don't have a name.", additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-2.0-flash', 'safety_ratings': []}, id='run-7aaf342c-eb45-4ff1-9d9b-96b3d1bb3138-0', usage_metadata={'input_tokens': 8, 'output_tokens': 20, 'total_tokens': 28, 'input_token_details': {'cache_read': 0}})

In [ ]:
class AgentState(TypedDict):
    query: str
    category: str
    sub_category : str
    response: str

In [54]:
def category (state:AgentState) -> AgentState:
    """Categorize the entrepreneur query. and categorize it into COM, HOP, CTO, COO.FIR"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "For example @COM what is my business growth? -> COM"
        "COM, HOP, , CTO, COO.FIR, Query: {query}"
    )
    chain = prompt | ChatOpenAI(temperature=0)
    category = chain.invoke({"query": state["query"]}).content
    print(category)
    return {"category": category}

In [ ]:
def COM (state: AgentState) -> AgentState:
    """""categorize into futher agents like Market Intelligence Agent,Positioning & Brand Strategy Agent, Acquisition Strategy Agent,
    Conversion & Onboarding Agent, Content & Ad Execution Agent,  Growth Performance & Analytics Agent,"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "Market Intelligence Agent,Positioning & Brand Strategy Agent, Acquisition Strategy Agent, Conversion & Onboarding Agent, Content & Ad Execution Agent,  Growth Performance & Analytics Agent, Query: {query}"
        "@COM What my business growth? -> growth"
        )
    chain = prompt | ChatOpenAI(temperature=0)
    sub_category = chain.invoke({"query": state["query"]}).content  
    return {"sub_category": sub_category}

def HOP (state: AgentState) -> AgentState:
    """categorize into futher agents like Product Development Agent, Product Design Agent, Product Marketing Agent,
    Product Management Agent, Product Analytics Agent"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "Product Development Agent, Product Design Agent, Product Marketing Agent, Product Management Agent, Product Analytics Agent, Query: {query}" )
    chain = prompt | ChatOpenAI(temperature=0)
    category = chain.invoke({"query": state["query"]}).content  
    return {"category": category}
def CTO (state: AgentState) -> AgentState:
    """categorize into futher agents like Software Development Agent, Software Design Agent, Software Testing Agent,
    Software Deployment Agent, Software Maintenance Agent"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "Software Development Agent, Software Design Agent, Software Testing Agent, Software Deployment Agent, Software Maintenance Agent, Query: {query}" )
    chain = prompt | ChatOpenAI(temperature=0)
    category = chain.invoke({"query": state["query"]}).content  
    return {"category": category}
def COO (state: AgentState) -> AgentState:
    """categorize into futher agents like Operations Management Agent, Supply Chain Management Agent, Quality Control Agent,
    Inventory Management Agent, Logistics Management Agent"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "Operations Management Agent, Supply Chain Management Agent, Quality Control Agent, Inventory Management Agent, Logistics Management Agent, Query: {query}" )
    chain = prompt | ChatOpenAI(temperature=0)
    category = chain.invoke({"query": state["query"]}).content  
    return {"category": category}
def FIR (state: AgentState) -> AgentState:
    """categorize into futher agents like Financial Planning Agent, Budgeting Agent, Accounting Agent,
    Financial Analysis Agent, Risk Management Agent"""
    prompt = ChatPromptTemplate.from_template(
        "Categorize the following user query into one of these category, the message has @ and mentioned a tag: "
        "Financial Planning Agent, Budgeting Agent, Accounting Agent, Financial Analysis Agent, Risk Management Agent, Query: {query}" )
    chain = prompt | ChatOpenAI(temperature=0)
    category = chain.invoke({"query": state["query"]}).content  
    return {"category": category}




In [ ]:
class Growth_analyzer_agent:
    def __init__(self, prompt):
        # Initialize model, prompt, and web search tool
        self.model = ChatGoogleGenerativeAI(model="gemini-2.0-flash", api_key=GOOGLE_API_KEY)
        self.prompt = prompt
        self.tools = [TavilySearchResults(
            max_results=5,
            search_depth="advanced",
            include_answer=True,)]
        
    def output (self, query):
        """"just a test"""
        
        print("this is an output from growth_analyzer_agent")    
        

In [57]:
def growth_performance_analyze (state: AgentState) -> AgentState:
   """analyze the growth performance and provide insights"""
   
   system_message = '''You are an expert advisor for foreign study options like location, university or subjects, based on IELTS/GRE scores.'''
   prompt = ChatPromptTemplate.from_messages([
                        ("system", system_message),
                        # MessagesPlaceholder("chat_history"),
                        ("human", "{input}"),
                        ("placeholder", "{agent_scratchpad}"),])
   growth = Growth_analyzer_agent(prompt)
   response = growth.output(state['query'])
   return {"response": "Growth performance analysis and insights"}

In [58]:
workflow = StateGraph(AgentState)
workflow.add_node("categorize_query",category)
workflow.add_node("COM",COM)
workflow.add_node("HOP",HOP)
workflow.add_node("CTO",CTO)
workflow.add_node("COO",COO)
workflow.add_node("FIR",FIR)
workflow.add_node("growth_performance_analyze",growth_performance_analyze)
workflow.add_edge(START, "categorize_query")
workflow.add_conditional_edges(
    "categorize_query",
    {
        "COM": COM,
        "HOP": HOP,
        "CTO": CTO,
        "COO": COO,
        "FIR": FIR
    }
)
workflow.add_edge("COM", "growth_performance_analyze")
workflow.add_edge("growth_performance_analyze",END)
workflow.set_entry_point("categorize_query")

app = workflow.compile()

In [64]:

query  = "@COM what is my business growth"
result = app.invoke({'query' : query})
print(result)

COM
{'query': '@COM what is my business growth', 'category': 'COM'}
